# Experiment-6 (Part 2): N-gram Statistical Language Model

## Implement a statistical language model from scratch using the N-gram approach to predict the next word in a sequence based on the preceding words in a text corpus.

### 1. Load and preprocess the corpus

In [1]:
import re
import math
from collections import Counter

with open("text_data.txt", "r", encoding="utf-8") as f:
    sentences = f.read().splitlines()

def tokenize(text):
    return re.findall(r"[a-z]+", text.lower())

tokenized = [tokenize(s) for s in sentences]

for s in tokenized:
    print(s)

['natural', 'language', 'processing', 'is', 'a', 'branch', 'of', 'artificial', 'intelligence']
['natural', 'language', 'processing', 'enables', 'computers', 'to', 'understand', 'human', 'language']
['machine', 'learning', 'is', 'a', 'branch', 'of', 'artificial', 'intelligence']
['deep', 'learning', 'is', 'a', 'part', 'of', 'machine', 'learning']
['language', 'models', 'learn', 'patterns', 'from', 'text', 'data']
['a', 'language', 'model', 'predicts', 'the', 'next', 'word', 'in', 'a', 'sentence']
['natural', 'language', 'processing', 'uses', 'language', 'models']
['artificial', 'intelligence', 'includes', 'machine', 'learning', 'and', 'deep', 'learning']


### 2. Build N-gram counts and convert to probabilities

- **Unigram model:** P(w)
- **Bigram model:** P(w | w_prev)
- **Trigram model:** P(w | w_prev, w_prev2)

Laplace (add-1) smoothing is applied so no probability is ever zero.

In [2]:
def build_ngram_model(tokenized_corpus, n, alpha=1.0):
    """Return (ngram_counts, context_counts, vocab, V) with add-alpha smoothing."""
    ngram_counts = Counter()
    context_counts = Counter()

    for sentence in tokenized_corpus:
        padded = ["<s>"] * (n - 1) + sentence + ["</s>"]
        for i in range(len(padded) - n + 1):
            ngram = tuple(padded[i : i + n])
            ngram_counts[ngram] += 1
            context_counts[ngram[:-1]] += 1

    vocab = sorted({w for sent in tokenized_corpus for w in sent} | {"<s>", "</s>"})
    V = len(vocab)
    return ngram_counts, context_counts, vocab, V


def prob(word, context, ngram_counts, context_counts, vocab, V, alpha=1.0):
    count_ctx = context_counts[context]
    return (ngram_counts[context + (word,)] + alpha) / (count_ctx + alpha * V)


models = {}
for n in (1, 2, 3):
    models[n] = build_ngram_model(tokenized, n)

for n, (ng, ctx, vocab, V) in models.items():
    print(f"{n}-gram: {len(ng)} unique n-grams, vocab size = {V}")

1-gram: 35 unique n-grams, vocab size = 36
2-gram: 53 unique n-grams, vocab size = 36
3-gram: 61 unique n-grams, vocab size = 36


### 3. Predict the next word (top-k candidates)

In [3]:
def predict_next(context_words, n, topk=5):
    ng, ctx_counts, vocab, V = models[n]
    if n == 1:
        context = ()
    else:
        needed = n - 1
        prefix = ["<s>"] * max(0, needed - len(context_words))
        context = tuple(prefix + context_words[-needed:])

    scores = []
    for w in vocab:
        if w == "<s>":
            continue
        scores.append((w, prob(w, context, ng, ctx_counts, vocab, V)))

    scores.sort(key=lambda x: -x[1])
    return scores[:topk]


def show_predictions(context_words):
    print(f"Context: {context_words}")
    for n in (1, 2, 3):
        label = {1: "Unigram", 2: "Bigram ", 3: "Trigram"}[n]
        preds = predict_next(context_words, n)
        formatted = ", ".join(f"{w} ({p:.3f})" for w, p in preds)
        print(f"  {label} -> {formatted}")
    print()


for ctx in [["natural", "language"], ["machine", "learning"], ["is", "a"], ["language", "models"], ["the"]]:
    show_predictions(ctx)

Context: ['natural', 'language']
  Unigram -> </s> (0.083), language (0.073), a (0.055), learning (0.055), artificial (0.037)
  Bigram  -> processing (0.093), models (0.070), </s> (0.047), model (0.047), a (0.023)
  Trigram -> processing (0.103), </s> (0.026), a (0.026), and (0.026), artificial (0.026)

Context: ['machine', 'learning']
  Unigram -> </s> (0.083), language (0.073), a (0.055), learning (0.055), artificial (0.037)
  Bigram  -> </s> (0.073), is (0.073), and (0.049), a (0.024), artificial (0.024)
  Trigram -> </s> (0.051), and (0.051), is (0.051), a (0.026), artificial (0.026)

Context: ['is', 'a']
  Unigram -> </s> (0.083), language (0.073), a (0.055), learning (0.055), artificial (0.037)
  Bigram  -> branch (0.073), language (0.049), part (0.049), sentence (0.049), </s> (0.024)
  Trigram -> branch (0.077), part (0.051), </s> (0.026), a (0.026), and (0.026)

Context: ['language', 'models']
  Unigram -> </s> (0.083), language (0.073), a (0.055), learning (0.055), artificial 

### 4. Complete sentences using the trained bigram model (greedy decoding)

In [4]:
def complete_sentence(prefix_words, n=2, max_words=8):
    ng, ctx_counts, vocab, V = models[n]
    generated = list(prefix_words)

    for _ in range(max_words):
        context = tuple(generated[-(n - 1):]) if n > 1 else ()
        best_word, best_p = None, -1.0
        for w in vocab:
            if w == "<s>":
                continue
            p = prob(w, context, ng, ctx_counts, vocab, V)
            if p > best_p:
                best_word, best_p = w, p
        if best_word == "</s>":
            break
        generated.append(best_word)

    return generated


for prefix in [["natural", "language"], ["machine", "learning"], ["a", "language", "model"]]:
    print(" ".join(prefix), "->", " ".join(complete_sentence(prefix)))

natural language -> natural language processing enables computers to understand human language processing
machine learning -> machine learning
a language model -> a language model predicts the next word in a branch of


### 5. Evaluate the model: average log-likelihood and perplexity on the corpus

Lower perplexity = better model.

In [5]:
def sentence_log_prob(sentence, n):
    ng, ctx_counts, vocab, V = models[n]
    padded = ["<s>"] * (n - 1) + sentence + ["</s>"]
    log_p = 0.0
    for i in range(len(padded) - n + 1):
        context = tuple(padded[i : i + n - 1])
        word = padded[i + n - 1]
        p = prob(word, context, ng, ctx_counts, vocab, V)
        log_p += math.log(p)
    return log_p


def perplexity(corpus, n):
    total_log_p = sum(sentence_log_prob(s, n) for s in corpus)
    total_tokens = sum(len(s) + 1 for s in corpus)  # include </s>
    return math.exp(-total_log_p / total_tokens)


print(f"{'Model':<10} {'Perplexity':>12}")
print("-" * 24)
for n in (1, 2, 3):
    print(f"{n}-gram{'':<5} {perplexity(tokenized, n):>12.3f}")

Model        Perplexity
------------------------
1-gram            27.584
2-gram            15.223
3-gram            16.478


### Observation
- Higher-order n-grams condition on more history and usually fit the training corpus better (lower perplexity), but need more data to estimate reliably.
- Laplace smoothing guarantees non-zero probabilities for unseen n-grams.
- Given a prefix such as *natural language*, the bigram/trigram models rank the most probable continuations from the corpus statistics.